# ISIC 2019 Bronze ingestion

This notebook creates the Bronze tables and file layout, downloads the ISIC 2019 archive and metadata into landing, extracts the source files, and writes source metadata into Bronze.

Fill in the two placeholder download URLs before running it.

In [ ]:
%run ../_setup_env


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import Request, urlopen
from uuid import uuid4
import hashlib
import json
import os
import shutil
import tarfile
import zipfile

from pyspark.sql import functions as F

from isic_pipeline.setup import build_layout, load_dataset_config, load_yaml_config

DATASET_CONFIG = load_dataset_config(CONFIG_ROOT / "datasets" / "isic_2019.yaml")
STORAGE_CONFIG = load_yaml_config(CONFIG_ROOT / "storage.yaml")
DATASET_CONFIG["landing_root"] = STORAGE_CONFIG["landing_root"]
DATASET_CONFIG["storage_root"] = STORAGE_CONFIG["storage_root"]
layout = build_layout(DATASET_CONFIG)
landing_paths = layout["landing_paths"]
bronze_paths = layout["bronze_paths"]
bronze_tables = layout["bronze_tables"]

COLLECTION_ID = 65
COLLECTION_NAME = "Challenge 2019: Training"
COLLECTION_SLUG = "challenge_2019_training"
ZIP_DOWNLOAD_URL = "https://api.isic-archive.com/api/v2/zip-download/url/"
METADATA_URL = "https://api.isic-archive.com/collections/65/metadata/"
ARCHIVE_FILENAME = f"{COLLECTION_SLUG}_{COLLECTION_ID}.zip"
METADATA_FILENAME = f"{COLLECTION_SLUG}_{COLLECTION_ID}_metadata.csv"
OVERWRITE_ARCHIVE = False
OVERWRITE_BRONZE_FILES = False
CHUNK_SIZE = 8 * 1024 * 1024
TMP_ROOT = Path("/local_disk0/tmp/isic_2019_bronze")
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}

print(layout)


## Helper functions

These helpers manage local and DBFS paths, file download, archive extraction, and lightweight metadata alignment.

In [ ]:
def dbfs_uri_to_local_path(dbfs_uri: str) -> Path:
    """Map a DBFS URI to the local DBFS mount path."""
    if dbfs_uri.startswith("dbfs:/"):
        return Path("/dbfs") / dbfs_uri.removeprefix("dbfs:/")
    return Path("/dbfs") / dbfs_uri.lstrip("/")


def require_configured_url(name: str, value: str) -> str:
    """Fail early when a required download URL is missing."""
    if not value:
        raise RuntimeError(f"Set {name} before running this notebook.")
    return value


def resolve_collection_archive_url(collection_id: int) -> str:
    """Resolve the current signed archive URL for a collection."""
    payload = json.dumps({"collections": str(collection_id)}).encode("utf-8")
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json",
    }

    csrf_token = os.environ.get("ISIC_ARCHIVE_CSRF_TOKEN")
    session_cookie = os.environ.get("ISIC_ARCHIVE_SESSION_COOKIE")
    if csrf_token:
        headers["X-CSRFToken"] = csrf_token
    if session_cookie:
        headers["Cookie"] = session_cookie

    request = Request(ZIP_DOWNLOAD_URL, data=payload, headers=headers, method="POST")
    try:
        with urlopen(request) as response:
            body = response.read().decode("utf-8")
    except HTTPError as exc:
        error_body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            "Failed to resolve collection archive URL from ISIC Archive. "
            f"HTTP {exc.code}: {error_body}. "
            "If the API requires authentication, set ISIC_ARCHIVE_SESSION_COOKIE and "
            "ISIC_ARCHIVE_CSRF_TOKEN in the notebook environment."
        ) from exc

    response_payload = json.loads(body)
    archive_url = response_payload.get("url") or response_payload.get("download_url")
    if not archive_url:
        raise RuntimeError(f"Unexpected zip-download response: {response_payload}")
    return archive_url


def stream_download(source_url: str, local_path: Path) -> Path:
    """Stream a remote file into local temporary storage."""
    request = Request(source_url, headers={"Accept": "application/octet-stream, application/zip, text/csv, */*"})
    with urlopen(request) as response, local_path.open("wb") as handle:
        while True:
            chunk = response.read(CHUNK_SIZE)
            if not chunk:
                break
            handle.write(chunk)
    return local_path


def extract_archive(archive_path: Path, destination_root: Path) -> Path:
    """Extract a zip or tar archive into a clean local folder."""
    if destination_root.exists():
        shutil.rmtree(destination_root)
    destination_root.mkdir(parents=True, exist_ok=True)
    suffixes = [suffix.lower() for suffix in archive_path.suffixes]
    if archive_path.suffix.lower() == ".zip":
        with zipfile.ZipFile(archive_path) as handle:
            handle.extractall(destination_root)
        return destination_root
    if ".tar" in suffixes or archive_path.suffix.lower() in {".tgz", ".gz"}:
        with tarfile.open(archive_path) as handle:
            handle.extractall(destination_root)
        return destination_root
    raise RuntimeError(f"Unsupported archive format: {archive_path.name}")


def sha256_file(path: Path) -> str:
    """Calculate a SHA-256 checksum for a file."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(CHUNK_SIZE)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def maybe_column(frame, candidates: list[str]):
    """Return the first matching column expression or a null string column."""
    for candidate in candidates:
        if candidate in frame.columns:
            return F.col(candidate)
    return F.lit(None).cast("string")


## Create Bronze objects

This step ensures the Bronze tables exist and that the landing and Bronze file directories are ready before any download starts.

In [ ]:
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS bronze.ingestion_runs (
      ingestion_run_id STRING,
      dataset_name STRING,
      source_version STRING,
      started_at TIMESTAMP,
      finished_at TIMESTAMP,
      status STRING,
      records_seen INT,
      records_written INT
    )
    USING DELTA
    """
)

spark.sql(
    """
    CREATE TABLE IF NOT EXISTS bronze.isic_2019_source_metadata (
      image_id STRING,
      source_uri STRING,
      patient_id STRING,
      lesion_id STRING,
      label_raw STRING,
      label_source STRING,
      label_normalized STRING,
      acquisition_date STRING,
      source_checksum STRING,
      ingestion_run_id STRING,
      ingested_at TIMESTAMP
    )
    USING DELTA
    """
)

dbutils.fs.mkdirs(landing_paths["dataset_archive"])
for path in bronze_paths.values():
    dbutils.fs.mkdirs(path)

display(spark.sql("SHOW TABLES IN bronze"))
display(dbutils.fs.ls(landing_paths["archives"]))
display(dbutils.fs.ls(bronze_paths["root"]))


## Configure Sources

This step validates that the archive and metadata download URLs have been filled in and defines the landing destinations.

In [ ]:
COLLECTION_ARCHIVE_URL = resolve_collection_archive_url(COLLECTION_ID)
METADATA_URL = require_configured_url("METADATA_URL", METADATA_URL)

archive_dbfs_path = f"{landing_paths['dataset_archive'].rstrip('/')}/{ARCHIVE_FILENAME}"
metadata_landing_dbfs_path = f"{landing_paths['dataset_archive'].rstrip('/')}/{METADATA_FILENAME}"
archive_dbfs_local_path = dbfs_uri_to_local_path(archive_dbfs_path)
metadata_landing_local_path = dbfs_uri_to_local_path(metadata_landing_dbfs_path)
archive_local_path = TMP_ROOT / ARCHIVE_FILENAME
metadata_local_path = TMP_ROOT / METADATA_FILENAME

print(f"Collection: {COLLECTION_NAME} ({COLLECTION_ID})")
print(f"Archive source: {COLLECTION_ARCHIVE_URL}")
print(f"Metadata source: {METADATA_URL}")
print(f"Archive landing path: {archive_dbfs_path}")
print(f"Metadata landing path: {metadata_landing_dbfs_path}")


## Download Sources

This step downloads the archive and metadata into local scratch space when needed and copies both files into the dataset landing folder.

In [ ]:
TMP_ROOT.mkdir(parents=True, exist_ok=True)
archive_dbfs_local_path.parent.mkdir(parents=True, exist_ok=True)
metadata_landing_local_path.parent.mkdir(parents=True, exist_ok=True)

archive_exists = archive_dbfs_local_path.exists()
if archive_exists and not OVERWRITE_ARCHIVE:
    shutil.copy2(archive_dbfs_local_path, archive_local_path)
    print(f"Using existing landing archive: {archive_dbfs_path}")
else:
    if archive_local_path.exists():
        archive_local_path.unlink()
    stream_download(COLLECTION_ARCHIVE_URL, archive_local_path)
    shutil.copy2(archive_local_path, archive_dbfs_local_path)
    print(f"Downloaded archive to landing: {archive_dbfs_path}")

metadata_exists = metadata_landing_local_path.exists()
if metadata_exists and not OVERWRITE_ARCHIVE:
    shutil.copy2(metadata_landing_local_path, metadata_local_path)
    print(f"Using existing landing metadata: {metadata_landing_dbfs_path}")
else:
    if metadata_local_path.exists():
        metadata_local_path.unlink()
    stream_download(METADATA_URL, metadata_local_path)
    shutil.copy2(metadata_local_path, metadata_landing_local_path)
    print(f"Downloaded metadata to landing: {metadata_landing_dbfs_path}")

display(dbutils.fs.ls(landing_paths["dataset_archive"]))


## Extract Source Files

This step unpacks the image archive and identifies the extracted image files that will be copied into the Bronze file area.

In [ ]:
extraction_root = extract_archive(archive_local_path, TMP_ROOT / "archive_contents")
image_files = sorted(
    [path for path in extraction_root.rglob("*") if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES]
)

if not image_files:
    raise RuntimeError("The collection archive did not contain any image files.")

print(f"Extracted image count: {len(image_files)}")
print(f"Metadata file: {metadata_local_path}")


## Copy Bronze Files

This step materializes extracted images and metadata into the governed Bronze file paths and records per-image checksums.

In [ ]:
images_target_root = dbfs_uri_to_local_path(bronze_paths["images"])
metadata_target_root = dbfs_uri_to_local_path(bronze_paths["metadata"])
images_target_root.mkdir(parents=True, exist_ok=True)
metadata_target_root.mkdir(parents=True, exist_ok=True)

copied_image_rows = []
existing_image_targets = 0
for image_path in image_files:
    target_path = images_target_root / image_path.name
    if target_path.exists() and not OVERWRITE_BRONZE_FILES:
        existing_image_targets += 1
    else:
        shutil.copy2(image_path, target_path)
    copied_image_rows.append(
        {
            "image_id": target_path.stem,
            "source_uri": f"{bronze_paths['images'].rstrip('/')}/{target_path.name}",
            "source_checksum": sha256_file(target_path),
        }
    )

metadata_target_path = metadata_target_root / metadata_local_path.name
if not metadata_target_path.exists() or OVERWRITE_BRONZE_FILES:
    shutil.copy2(metadata_local_path, metadata_target_path)

print(f"Images already present and reused: {existing_image_targets}")
print(f"Bronze metadata file: {metadata_target_path}")


## Write Bronze Tables

This step loads the raw metadata file, aligns it with the copied images, merges the source rows into the Bronze table, and appends an ingestion run record.

In [ ]:
metadata_dbfs_path = f"{bronze_paths['metadata'].rstrip('/')}/{metadata_target_path.name}"
raw_metadata_df = spark.read.option("header", True).csv(metadata_dbfs_path)

image_manifest_df = spark.createDataFrame(copied_image_rows).dropDuplicates(["image_id"])
image_id_expr = maybe_column(raw_metadata_df, ["image", "image_id", "isic_id", "id"])
source_metadata_df = (
    raw_metadata_df
    .withColumn("image_id", image_id_expr.cast("string"))
    .join(image_manifest_df, on="image_id", how="left")
    .select(
        F.col("image_id"),
        F.col("source_uri"),
        maybe_column(raw_metadata_df, ["patient_id", "patient"]).alias("patient_id"),
        maybe_column(raw_metadata_df, ["lesion_id", "lesion"]).alias("lesion_id"),
        maybe_column(raw_metadata_df, ["dx", "diagnosis", "label"]).alias("label_raw"),
        F.lit(metadata_target_path.name).alias("label_source"),
        F.lit(None).cast("string").alias("label_normalized"),
        maybe_column(raw_metadata_df, ["acquisition_date", "date", "image_date"]).alias("acquisition_date"),
        F.col("source_checksum"),
    )
    .where(F.col("image_id").isNotNull())
)

ingestion_run_id = f"bronze-{COLLECTION_ID}-{uuid4().hex[:12]}"
started_at = datetime.now(timezone.utc)
source_metadata_df = (
    source_metadata_df
    .withColumn("ingestion_run_id", F.lit(ingestion_run_id))
    .withColumn("ingested_at", F.lit(started_at))
)

source_metadata_df.createOrReplaceTempView("staged_isic_2019_source_metadata")
spark.sql(
    f"""
    MERGE INTO {bronze_tables['source_metadata']} AS target
    USING staged_isic_2019_source_metadata AS source
    ON target.image_id = source.image_id
    WHEN MATCHED THEN UPDATE SET
      target.source_uri = source.source_uri,
      target.patient_id = source.patient_id,
      target.lesion_id = source.lesion_id,
      target.label_raw = source.label_raw,
      target.label_source = source.label_source,
      target.label_normalized = source.label_normalized,
      target.acquisition_date = source.acquisition_date,
      target.source_checksum = source.source_checksum,
      target.ingestion_run_id = source.ingestion_run_id,
      target.ingested_at = source.ingested_at
    WHEN NOT MATCHED THEN INSERT *
    """
)

records_written = source_metadata_df.count()
finished_at = datetime.now(timezone.utc)
run_row = [{
    "ingestion_run_id": ingestion_run_id,
    "dataset_name": DATASET_CONFIG["dataset_name"],
    "source_version": DATASET_CONFIG["source_version"],
    "started_at": started_at,
    "finished_at": finished_at,
    "status": "success",
    "records_seen": records_written,
    "records_written": records_written,
}]
spark.createDataFrame(run_row).write.mode("append").saveAsTable(bronze_tables["ingestion_runs"])

print(f"Ingestion run id: {ingestion_run_id}")
print(f"Records written: {records_written}")


## Review Outputs

This final step shows the landed archive, Bronze file paths, and the most recent Bronze table contents for inspection.

In [ ]:
print("Landing archive contents:")
display(dbutils.fs.ls(landing_paths["dataset_archive"]))

print("Bronze metadata path contents:")
display(dbutils.fs.ls(bronze_paths["metadata"]))

print("Bronze images path sample:")
display(dbutils.fs.ls(bronze_paths["images"]))

print("Recent ingestion runs:")
display(spark.table(bronze_tables["ingestion_runs"]).orderBy("started_at", ascending=False))

print("Source metadata sample:")
display(spark.table(bronze_tables["source_metadata"]))
